# Candidates that need no retraining

Scores the v5 checkpoints and their weight average on validation, picks the best on validation, scores it once on test (recording per-clip confidence and hand detection), runs the gating analysis, and sweeps decoding settings on validation.

In [ ]:
BRANCH = "phase-4-accuracy"
!git clone -q --branch {BRANCH} https://github.com/sohenpatel22/ASL-Fingerspelling-Recognition.git /tmp/repo
%cd /tmp/repo
!pip install -q -e .

In [ ]:
import os, zipfile
from pathlib import Path

def walk(root="/kaggle/input", max_depth=5):
    for base, dirs, files in os.walk(root):
        dirs[:] = [d for d in dirs if d not in ("npy_train", "npy_supp", "competitions")]
        if base[len(root):].count(os.sep) >= max_depth:
            dirs[:] = []
        yield Path(base), dirs, files

def find_dir(name, root="/kaggle/input"):
    for base, dirs, _ in os.walk(root):
        if name in dirs:
            return Path(base, name)
        if base[len(root):].count(os.sep) >= 5:
            dirs[:] = []
    raise FileNotFoundError(name)

COMP = next(p for p in (Path("/kaggle/input/competitions/asl-fingerspelling"), Path("/kaggle/input/asl-fingerspelling")) if p.exists())
try:
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train"), find_dir("npy_supp")
except FileNotFoundError:
    for archive in [p for p in Path("/kaggle/input").rglob("*.zip") if "competitions" not in str(p)]:
        with zipfile.ZipFile(archive) as z:
            z.extractall("/tmp/npy")
    NPY_TRAIN, NPY_SUPP = find_dir("npy_train", "/tmp/npy"), find_dir("npy_supp", "/tmp/npy")
DATA = f"data.comp_dir={COMP} data.npy_train={NPY_TRAIN} data.npy_supp={NPY_SUPP}"

found = {f: base / f for base, _, files in walk() for f in files if f.startswith("asl_transformer_v5_") and f.endswith(".pth")}
print("checkpoints found on kaggle:", sorted(found))

In [ ]:
from asl.checkpoint import fetch_hub_file

CANDS = {"hub_best": str(fetch_hub_file("hf:SohenP/asl-fingerspelling-conformer/asl_v5_best.pth"))}
for tag in ("best", "final", "last"):
    if f"asl_transformer_v5_{tag}.pth" in found:
        CANDS[f"k_{tag}"] = str(found[f"asl_transformer_v5_{tag}.pth"])
soup_inputs = [v for k, v in CANDS.items() if k.startswith("k_")]
if len(soup_inputs) >= 2:
    !python -m asl.checkpoint average {" ".join(soup_inputs)} --out /kaggle/working/soup.pth --allow-unsafe
    CANDS["soup"] = "/kaggle/working/soup.pth"
print(CANDS)

In [ ]:
import json

val = {}
for name, path in CANDS.items():
    !asl-eval --checkpoint {path} --allow-unsafe --config configs/finetune_v5.yaml --split val --max-samples 1000 --sample-seed 0 --out /kaggle/working/val_{name}.json {DATA}
    val[name] = json.load(open(f"/kaggle/working/val_{name}.json"))["cer"]
best = min(val, key=val.get)
print("validation CER by candidate:", {k: round(v, 4) for k, v in val.items()})
print("best on validation:", best)

In [ ]:
!asl-eval --checkpoint {CANDS[best]} --allow-unsafe --config configs/finetune_v5.yaml --split test --max-samples 3000 --sample-seed 0 --out /kaggle/working/test_best.json --details-out /kaggle/working/test_best_details.json {DATA}
assert Path("/kaggle/working/test_best.json").exists(), "test scoring failed: read the output above"
print(open("/kaggle/working/test_best.json").read())

In [ ]:
from asl.gating import summarize_gating

records = json.load(open("/kaggle/working/test_best_details.json"))["predictions"]
gating = summarize_gating(records)
json.dump(gating, open("/kaggle/working/gating.json", "w"), indent=1)
print("share of clips with CER >= 0.9:", round(gating["catastrophic_share"], 3))
print("AUROC for spotting those clips:", {k: round(v, 3) for k, v in gating["auroc"].items()})
for key, curve in gating["curves"].items():
    print()
    print("error if only the most-trusted clips (by " + key + ") are accepted:")
    for p in curve[::2] + [curve[-1]]:
        print(f"  coverage {p['coverage']:.0%}  CER {p['cer']:.3f}  (threshold {p['threshold']:.2f})")


In [ ]:
sweep = {"greedy": "decode.beam_width=1", "beam5_lp0.4": "decode.beam_width=5 decode.length_penalty=0.4",
         "beam5_lp0.8": "decode.beam_width=5 decode.length_penalty=0.8", "beam5_lp1.0": "decode.beam_width=5 decode.length_penalty=1.0"}
dec = {}
for tag, overrides in sweep.items():
    !asl-eval --checkpoint {CANDS[best]} --allow-unsafe --config configs/finetune_v5.yaml --split val --max-samples 1000 --sample-seed 0 --out /kaggle/working/dec_{tag}.json {DATA} {overrides}
    dec[tag] = json.load(open(f"/kaggle/working/dec_{tag}.json"))["cer"]
dec["beam5_lp0.6 (current)"] = val[best]
print("validation CER by decoding setting:", {k: round(v, 4) for k, v in dec.items()})
json.dump({"candidates_val": val, "decoding_val": dec, "best": best}, open("/kaggle/working/summary.json", "w"), indent=1)